# Chapter 5 — Pre-Training

**Goal:** Train the GPT model to predict the next token on unlabeled text.

### Key concepts
- Cross-entropy loss for next-token prediction
- AdamW optimizer
- Gradient clipping
- Perplexity as evaluation metric
- Training vs. validation loss curves

In [ ]:
import sys, torch
import matplotlib.pyplot as plt
sys.path.insert(0, '..')
from src.model.gpt import GPT, GPTConfig
from src.data.dataset import GPTDataset
from src.training.pretrain import compute_loss, evaluate, train
from torch.utils.data import DataLoader, random_split

## 1. Prepare Data

In [ ]:
# In a real run, replace with tokenized text from a real corpus:
# from src.data.tokenizer import BPETokenizer
# tok = BPETokenizer()
# with open('../data/samples/sample_text.txt') as f:
#     token_ids = tok.encode(f.read())

# Dummy data for demonstration:
torch.manual_seed(0)
token_ids = torch.randint(0, 50257, (2000,)).tolist()

CONTEXT = 32
ds = GPTDataset(token_ids, context_length=CONTEXT, stride=CONTEXT//2)
train_size = int(0.9 * len(ds))
train_ds, val_ds = random_split(ds, [train_size, len(ds) - train_size])

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=8)
print(f'Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')

## 2. Initialise Model & Optimizer

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

cfg = GPTConfig(vocab_size=50257, context_length=CONTEXT,
                embed_dim=128, num_heads=4, num_layers=2)
model = GPT(cfg).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
print(f'Model parameters: {model.num_parameters():,}')

## 3. Training Loop

In [ ]:
history = train(
    model, train_loader, val_loader, optimizer, DEVICE,
    num_epochs=3, eval_every=10, save_path='../data/gpt_pretrained.pt'
)

## 4. Plot Loss Curves

In [ ]:
if history['step']:
    plt.figure(figsize=(8, 4))
    plt.plot(history['step'], history['train_loss'], label='Train loss')
    plt.plot(history['step'], history['val_loss'],   label='Val loss', linestyle='--')
    plt.xlabel('Step'); plt.ylabel('Loss')
    plt.title('Pre-Training Loss Curves')
    plt.legend(); plt.grid(True); plt.show()

## 5. Exercises
1. Replace dummy token IDs with a real text file and compare loss.
2. What is the perplexity of an untrained model on a vocab of 50,257 tokens?
3. Experiment with learning rate schedules (warmup + cosine decay).